# General

All project work in IND320 will result in personal hand-ins and online apps.

### 1. Jupyter Notebook

A Jupyter Notebook run locally on your computer, later with access to online and local databases.

This will be your basic development and documentation platform.

#### Required content

- A brief description of AI usage.
- A 300–500-word log describing the compulsory work, including both Jupyter Notebook and Streamlit experience.
- Links to your public GitHub repository and Streamlit app for the compulsory work.

#### Documentation and submission

- Document headings should be clear and usable for navigation during development.
- All code blocks must include enough comments to be understandable and reproducible if someone inherits your project.
- All code blocks must be run before exporting to PDF so that messages and plots are shown.
- Add the `.ipynb` file to the GitHub repository containing your Streamlit project.

### 2. Streamlit App

A Streamlit app running at `https://[yourproject].streamlit.app/`.

- This will be an online version of the project, accessing data exported to CSV format and, later, an online database.
- The code must be hosted on GitHub.
- The code must include relevant comments from the Jupyter Notebook and additional comments regarding Streamlit usage.

#### Project Structure and Collaboration

- There are four parts in the project work, building on each other and resulting in a final portfolio and app to be presented at the end of the semester.
- Cooperation is applauded, and the use of AI tools is encouraged.

## Tasks

### GitHub and Streamlit.app accounts

- GitHub: https://github.com/danielytsan/IND320APP
- Steamlit: https://ind320app-s2bytfwvqxbzmwkxrm2aze.streamlit.app

# Presentation link

 Due to the video file being to large for github I have choosen to upload it to youtube. Use this link to access it:

https://youtu.be/NM3GvSNXUs8


## AI usage
AI has been used during development of the content provided for this app and Jupyter Notebook

The model I have been using is ChatGPT-6.1 Sol to:
- debugging and suggestion for code with error messages
- learning streamlit and different function
- learing to use api keys and how to format the files for it. Specially for how to use the token on Entso-e.
- help with using spark
- Used for helping with merge conflict
- some guidens on explaining the concept of the task

I have also started to explore suggestions agentic coding is giving me with copilot

# Jupyter Notebook

* Make a function for downloading Euro and USD currency exchange rates from the Norwegian Bank (NB) currency API and show that it works.

* Investigate possible timezone differences between Magasinstatistikk, ENTSO-E, OpenMeteo and NB and report your findings, plus solutions if needed.

* Using the River framework, make a function for SARIMAX or a lag-based Machine Learning with predictions of the coming months' fillings for a single region (according to Magasinstatistikk), letting relevant model parameters, length of history and horizon, and selection of data sources be inputs to the function.

    * Choose a reasonable time resolution for your input data so the function doesn't use an unreasonable amount of time while still performing well. Include findings in the report.

    * Test the function for various settings, report your findings (how you tested and your experiences) and let the best solution be the default setting when used in Streamlit.

* Include the sketch mentioned in "Streamlit app, refactoring -> Structure and navigation" below.

In [1]:
#importing libraris
import pandas as pd
import requests

In [7]:
def exchange_rates(observations = 60):
    url = "https://data.norges-bank.no/api/data/EXR/B.USD+EUR.NOK.SP?format=sdmx-json&lastNObservations=60&locale=no"
    response = requests.get(
        url,
        params = {
            "format": "sdmx-json",
            "lastNObservation": observations,
            "locale": "no",
        },
        timeout = 30,
    )
    response.raise_for_status()
    payload = response.json()
    data = payload.get("data", payload)

    #reading currency labels and dates
    dimensions = data["structure"]["dimensions"]
    currency_position = next(
        i for i, dimension in enumerate(dimensions["series"])
        if dimension["id"] == "BASE_CUR"
    )
    currencies = dimensions["series"][currency_position]["values"]
    dates = dimensions["observation"][0]["values"]

    #match each observation to its currency and date
    records = []
    for key, series in data["dataSets"][0]["series"].items():
        currency_index = int(key.split(":")[currency_position])
        currency = currencies[currency_index]["id"]

        for date_index, values in series["observations"].items():
            records.append({
                "Date": dates[int(date_index)]["id"],
                "Currency": currency,
                "Rate": values[0],
            })

    #creating table with dates as rows and currencies as column
    rates = pd.DataFrame(records)
    rates["Date"] = pd.to_datetime(rates["Date"])
    rates["Rate"] = pd.to_numeric(rates["Rate"], errors = "raise")
    rates = rates.pivot(index = "Date", columns = "Currency", values = "Rate")
    rates.columns.name = None

    return rates.sort_index()[["EUR", "USD"]]

#running the function
rates = exchange_rates()
assert rates.count().to_dict() == {"EUR": 60, "USD": 60}
assert (rates > 0).all().all()
assert rates.index.is_unique and rates.index.is_monotonic_increasing
print(rates.tail().to_string(float_format = "{:.4f}".format))


               EUR    USD
Date                     
2026-11-16 10.5740 9.6005
2026-11-17 10.5755 9.6237
2026-11-18 10.6045 9.6686
2026-11-19 10.6475 9.6866
2026-11-20 10.7180 9.7525
